# F0 Contour Extraction — pyin vs IDTAP Evaluation

Extract F0 with **librosa.pyin**, compare against IDTAP ground-truth contours, sweep parameters, and decide whether a pitch-contour approach is viable.

In [9]:
from __future__ import annotations

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio, display
from idtap import Piece, SwaraClient
from idtap.classes.trajectory import Trajectory

PIECE_ID = "6824de49abc4705438ce918b"
INST = 0
STRING_IDX = 0
SR = 22050
HOP_LENGTH = 512
FMIN = 75
FMAX = 2400
AUDIO_FORMAT = "wav"
SILENT_TRAJECTORY_ID = 12
PREVIEW_COUNT = 10

# Go/no-go thresholds (tuned for trajectory shape classification)
MAE_ABANDON_CENTS = 60.0      # abandon if best mean MAE exceeds this
MAE_CAUTION_CENTS = 40.0      # proceed with caution between this and ABANDON
VOICED_MIN_PCT = 55.0         # abandon if best mean voiced % on non-silent trajs is below this
GOOD_TRAJ_FRACTION = 0.60     # fraction of non-silent trajs with MAE < 50 cents to proceed confidently
GOOD_TRAJ_MAE_CENTS = 50.0

# Parameter grid to sweep (baseline first)
PARAM_GRID: list[dict[str, int | float]] = [
    {"hop_length": 512, "fmin": 75, "fmax": 2400, "frame_length": 2048},
    {"hop_length": 256, "fmin": 75, "fmax": 2400, "frame_length": 1024},
    {"hop_length": 1024, "fmin": 75, "fmax": 2400, "frame_length": 4096},
    {"hop_length": 512, "fmin": 75, "fmax": 1200, "frame_length": 2048},
    {"hop_length": 512, "fmin": 75, "fmax": 1800, "frame_length": 2048},
    {"hop_length": 512, "fmin": 65, "fmax": 2400, "frame_length": 2048},
    {"hop_length": 512, "fmin": 100, "fmax": 2400, "frame_length": 2048},
    {"hop_length": 256, "fmin": 100, "fmax": 1800, "frame_length": 1024},
    {"hop_length": 1024, "fmin": 65, "fmax": 1200, "frame_length": 4096},
]

OUTPUT_DIR = REPO_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

LABEL_COLORS = {
    0: "#4C72B0",
    1: "#DD8452",
    2: "#55A868",
    3: "#C44E52",
    "silent": "#BBBBBB",
}

In [10]:
def load_piece_and_audio(client: SwaraClient, piece_id: str) -> tuple[Piece, Path]:
    piece = Piece.from_json(client.get_piece(piece_id))
    audio_path = client.download_and_save_transcription_audio(
        piece,
        format=AUDIO_FORMAT,
        filepath=str(OUTPUT_DIR),
    )
    if audio_path is None:
        raise RuntimeError(f"No audio could be downloaded for piece {piece_id}")
    return piece, Path(audio_path)


def build_traj_selections(piece: Piece) -> list[dict[str, object]]:
    trajectories = piece.all_trajectories(inst=INST, string_idx=STRING_IDX)
    start_times = piece.traj_start_times(inst=INST, string_idx=STRING_IDX)
    selections: list[dict[str, object]] = []
    for idx, traj in enumerate(trajectories):
        if idx >= len(start_times):
            break
        traj_start = float(start_times[idx])
        selections.append(
            {
                "traj": traj,
                "index": idx,
                "start": traj_start,
                "end": traj_start + float(traj.dur_tot),
            }
        )
    return selections


def trajectory_label(traj: Trajectory) -> int | str:
    if traj.id == SILENT_TRAJECTORY_ID:
        return "silent"
    if traj.id in {0, 1, 2, 3}:
        return traj.id
    if traj.id == 6:
        return 1
    if traj.id == 4:
        return 2
    if traj.id == 5:
        return 3
    return "other"


def idtap_contour(traj: Trajectory, n: int = 200) -> tuple[np.ndarray, np.ndarray]:
    if traj.id == SILENT_TRAJECTORY_ID:
        return np.array([]), np.array([])
    xs = np.linspace(0.0, 1.0, n, endpoint=False)
    times = xs * float(traj.dur_tot)
    freqs = np.array([traj.compute(float(x), log_scale=False) for x in xs])
    return times, freqs


def extract_pyin(
    y: np.ndarray,
    sr: int,
    *,
    fmin: float = FMIN,
    fmax: float = FMAX,
    hop_length: int = HOP_LENGTH,
    frame_length: int | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    kwargs: dict[str, int | float] = {
        "fmin": fmin,
        "fmax": fmax,
        "sr": sr,
        "hop_length": hop_length,
    }
    if frame_length is not None:
        kwargs["frame_length"] = frame_length
    f0, voiced_flag, _ = librosa.pyin(y, **kwargs)
    times = librosa.times_like(f0, sr=sr, hop_length=hop_length)
    return times, f0, voiced_flag.astype(bool)


def evaluate_trajectories(
    times: np.ndarray,
    f0: np.ndarray,
    traj_selections: list[dict[str, object]],
    tonic_hz: float,
) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for sel in traj_selections:
        traj = sel["traj"]
        start = float(sel["start"])
        duration = float(traj.dur_tot)
        label = trajectory_label(traj)
        seg_times, seg_f0 = slice_f0(times, f0, start, duration)
        voiced_mask = np.isfinite(seg_f0) & (seg_f0 > 0)
        row: dict[str, object] = {
            "traj_index": sel["index"],
            "idtap_name": traj.name_,
            "label": label,
            "duration": duration,
            "voiced_pct": voiced_pct(seg_f0),
            "mae_cents": np.nan,
        }
        if traj.id != SILENT_TRAJECTORY_ID:
            idtap_times, idtap_freqs = idtap_contour(traj)
            ref = resample_contour_to_times(seg_times, idtap_times, idtap_freqs)
            row["mae_cents"] = mae_cents(seg_f0, ref, tonic_hz, voiced_mask)
        rows.append(row)
    return pd.DataFrame(rows)


def summarize_eval(per_traj: pd.DataFrame) -> dict[str, float]:
    non_silent = per_traj[per_traj["label"] != "silent"]
    good = non_silent["mae_cents"] < GOOD_TRAJ_MAE_CENTS
    return {
        "mean_mae_cents": float(non_silent["mae_cents"].mean()),
        "median_mae_cents": float(non_silent["mae_cents"].median()),
        "mean_voiced_pct": float(non_silent["voiced_pct"].mean()),
        "good_traj_fraction": float(good.mean()) if len(good) else 0.0,
        "n_non_silent": float(len(non_silent)),
    }


def config_label(params: dict[str, int | float]) -> str:
    return (
        f"hop={params['hop_length']} fmin={params['fmin']} "
        f"fmax={params['fmax']} frame={params.get('frame_length', 'auto')}"
    )


def decide_viability(best: dict[str, float]) -> tuple[str, str]:
    mean_mae = best["mean_mae_cents"]
    voiced = best["mean_voiced_pct"]
    good_frac = best["good_traj_fraction"]
    if mean_mae > MAE_ABANDON_CENTS or voiced < VOICED_MIN_PCT:
        return (
            "ABANDON",
            "pyin tracks too poorly vs IDTAP even after tuning — "
            "pitch-contour classification is unlikely to work.",
        )
    if mean_mae > MAE_CAUTION_CENTS or good_frac < GOOD_TRAJ_FRACTION:
        return (
            "CAUTION",
            "pyin is borderline — consider per-trajectory refinement or a different representation.",
        )
    return (
        "PROCEED",
        "pyin is accurate enough to pursue contour-based trajectory classification.",
    )


def slice_f0(
    times: np.ndarray,
    f0: np.ndarray,
    start: float,
    duration: float,
) -> tuple[np.ndarray, np.ndarray]:
    end = start + duration
    mask = (times >= start) & (times < end)
    return times[mask] - start, f0[mask]


def freq_to_cents(f: np.ndarray, tonic_hz: float) -> np.ndarray:
    out = np.full_like(f, np.nan, dtype=float)
    valid = np.isfinite(f) & (f > 0)
    out[valid] = 1200.0 * np.log2(f[valid] / tonic_hz)
    return out


def resample_contour_to_times(
    ref_times: np.ndarray,
    contour_times: np.ndarray,
    contour_freqs: np.ndarray,
) -> np.ndarray:
    if contour_times.size == 0 or ref_times.size == 0:
        return np.full_like(ref_times, np.nan, dtype=float)
    return np.interp(ref_times, contour_times, contour_freqs, left=np.nan, right=np.nan)


def mae_cents(
    est_freqs: np.ndarray,
    ref_freqs: np.ndarray,
    tonic_hz: float,
    mask: np.ndarray | None = None,
) -> float:
    if mask is None:
        mask = np.ones_like(est_freqs, dtype=bool)
    valid = (
        mask
        & np.isfinite(est_freqs)
        & np.isfinite(ref_freqs)
        & (est_freqs > 0)
        & (ref_freqs > 0)
    )
    if not np.any(valid):
        return float("nan")
    est_cents = freq_to_cents(est_freqs[valid], tonic_hz)
    ref_cents = freq_to_cents(ref_freqs[valid], tonic_hz)
    return float(np.mean(np.abs(est_cents - ref_cents)))


def voiced_pct(f0: np.ndarray, mask: np.ndarray | None = None) -> float:
    if mask is None:
        mask = np.ones_like(f0, dtype=bool)
    subset = f0[mask]
    if subset.size == 0:
        return float("nan")
    valid = np.isfinite(subset) & (subset > 0)
    return 100.0 * float(np.mean(valid))

In [11]:
client = SwaraClient()
piece, audio_path = load_piece_and_audio(client, PIECE_ID)
y_full, sr = librosa.load(audio_path, sr=SR, mono=True)
traj_selections = build_traj_selections(piece)
tonic_hz = float(piece.raga.fundamental) if piece.raga and piece.raga.fundamental else None

print(f"Title: {piece.title}")
print(f"Piece ID: {PIECE_ID}")
print(f"Trajectories: {len(traj_selections)}")
print(f"Tonic: {tonic_hz:.2f} Hz" if tonic_hz else "Tonic: unknown")
print(f"Audio duration: {len(y_full) / sr:.2f}s")
print(f"Audio path: {audio_path}")

/Users/raymondzhang/miniconda3/envs/idtap/lib/python3.10/site-packages/idtap/classes/trajectory.py:388: UserWarning: Vocal parameters provided but instrumentation is Sarangi. Vocal parameters are typically used with Vocal_M or Vocal_F instruments.
  warnings.warn(f"Vocal parameters provided but instrumentation is {instrumentation.name}. "


Title: Abdul Latif Khan - Basant Mukrahi
Piece ID: 6824de49abc4705438ce918b
Trajectories: 535
Tonic: 338.00 Hz
Audio duration: 591.99s
Audio path: output/Abdul Latif Khan - Basant Mukrahi_6824de49abc4705438ce918b.wav


In [12]:
if tonic_hz is None:
    raise RuntimeError("Tonic is required for cents-based evaluation.")

sweep_rows: list[dict[str, object]] = []
cached_tracks: dict[str, tuple[np.ndarray, np.ndarray, np.ndarray]] = {}

for i, params in enumerate(PARAM_GRID, start=1):
    label = config_label(params)
    print(f"[{i}/{len(PARAM_GRID)}] Running pyin: {label}")
    times, f0, voiced = extract_pyin(y_full, sr, **params)
    cached_tracks[label] = (times, f0, voiced)
    per_traj = evaluate_trajectories(times, f0, traj_selections, tonic_hz)
    stats = summarize_eval(per_traj)
    sweep_rows.append({"config": label, **params, **stats})

sweep_df = pd.DataFrame(sweep_rows).sort_values("mean_mae_cents")
display(sweep_df)

best_row = sweep_df.iloc[0]
best_label = str(best_row["config"])
best_config = {
    "hop_length": int(best_row["hop_length"]),
    "fmin": float(best_row["fmin"]),
    "fmax": float(best_row["fmax"]),
    "frame_length": int(best_row["frame_length"]),
}

best_times, best_f0, best_voiced = cached_tracks[best_label]
best_per_traj = evaluate_trajectories(best_times, best_f0, traj_selections, tonic_hz)
best_stats = summarize_eval(best_per_traj)
decision, rationale = decide_viability(best_stats)

print("\n=== Best config ===")
print(config_label(best_config))
print(
    f"  mean MAE:   {best_stats['mean_mae_cents']:.1f} cents\n"
    f"  median MAE: {best_stats['median_mae_cents']:.1f} cents\n"
    f"  mean voiced: {best_stats['mean_voiced_pct']:.1f}%\n"
    f"  good trajs:  {100 * best_stats['good_traj_fraction']:.1f}% under {GOOD_TRAJ_MAE_CENTS:.0f} cents"
)
print(f"\n=== Decision: {decision} ===")
print(rationale)

[1/9] Running pyin: hop=512 fmin=75 fmax=2400 frame=2048
[2/9] Running pyin: hop=256 fmin=75 fmax=2400 frame=1024
[3/9] Running pyin: hop=1024 fmin=75 fmax=2400 frame=4096
[4/9] Running pyin: hop=512 fmin=75 fmax=1200 frame=2048
[5/9] Running pyin: hop=512 fmin=75 fmax=1800 frame=2048
[6/9] Running pyin: hop=512 fmin=65 fmax=2400 frame=2048
[7/9] Running pyin: hop=512 fmin=100 fmax=2400 frame=2048
[8/9] Running pyin: hop=256 fmin=100 fmax=1800 frame=1024
[9/9] Running pyin: hop=1024 fmin=65 fmax=1200 frame=4096


,config,hop_length,fmin,fmax,frame_length,mean_mae_cents,median_mae_cents,mean_voiced_pct,good_traj_fraction,n_non_silent
7,hop=256 fmin=100 fmax=1800 frame=1024,256,100,1800,1024,205.514857,48.602109,87.264953,0.503356,447.0
6,hop=512 fmin=100 fmax=2400 frame=2048,512,100,2400,2048,233.893794,48.439194,88.192967,0.494407,447.0
1,hop=256 fmin=75 fmax=2400 frame=1024,256,75,2400,1024,308.090644,53.513382,86.925179,0.480984,447.0
3,hop=512 fmin=75 fmax=1200 frame=2048,512,75,1200,2048,366.854639,53.778984,87.363612,0.478747,447.0
4,hop=512 fmin=75 fmax=1800 frame=2048,512,75,1800,2048,370.726746,54.858578,87.814895,0.478747,447.0
0,hop=512 fmin=75 fmax=2400 frame=2048,512,75,2400,2048,371.518400,55.938173,88.119728,0.478747,447.0
5,hop=512 fmin=65 fmax=2400 frame=2048,512,65,2400,2048,388.834663,56.741893,88.138215,0.469799,447.0
2,hop=1024 fmin=75 fmax=2400 frame=4096,1024,75,2400,4096,459.379038,68.905659,89.794232,0.427293,447.0
8,hop=1024 fmin=65 fmax=1200 frame=4096,1024,65,1200,4096,460.047787,64.137914,89.115148,0.434004,447.0



=== Best config ===
hop=256 fmin=100.0 fmax=1800.0 frame=1024
  mean MAE:   205.5 cents
  median MAE: 48.6 cents
  mean voiced: 87.3%
  good trajs:  50.3% under 50 cents

=== Decision: ABANDON ===
pyin tracks too poorly vs IDTAP even after tuning — pitch-contour classification is unlikely to work.


In [14]:
by_label = (
    best_per_traj[best_per_traj["label"] != "silent"]
    .groupby("label", dropna=False)
    .agg(
        count=("mae_cents", "size"),
        mean_mae_cents=("mae_cents", "mean"),
        median_mae_cents=("mae_cents", "median"),
        mean_voiced_pct=("voiced_pct", "mean"),
    )
    .sort_index()
)
print("Per-label stats (best config):")
display(by_label)

if decision == "ABANDON":
    print("\nSkipping visualizations — pyin contour method abandoned.")
else:
    pyin_times, pyin_f0, pyin_voiced = best_times, best_f0, best_voiced
    summary = best_per_traj.rename(
        columns={"voiced_pct": "pyin_voiced_pct", "mae_cents": "pyin_mae_cents"}
    )
    print(f"\nUsing best config for remaining cells: {config_label(best_config)}")

TypeError: '<' not supported between instances of 'str' and 'int'

In [15]:
if decision == "ABANDON":
    print("Visualization skipped.")
else:
    from matplotlib.patches import Patch

    fig, ax = plt.subplots(figsize=(16, 5))

    for sel in traj_selections:
        traj = sel["traj"]
        label = trajectory_label(traj)
        color = LABEL_COLORS.get(label, "#CCCCCC")
        ax.axvspan(sel["start"], sel["end"], color=color, alpha=0.15, linewidth=0)

    ax.plot(pyin_times, pyin_f0, color="#E45756", linewidth=0.8, alpha=0.85, label="pyin")

    ax.set_ylabel("Frequency (Hz)")
    ax.set_xlabel("Time (s)")
    ax.set_title(f"{piece.title} — full-song F0 ({config_label(best_config)})")
    ax.set_ylim(FMIN, min(FMAX, 1200))

    tracker_handles, tracker_labels = ax.get_legend_handles_labels()
    label_patches = [
        Patch(facecolor=LABEL_COLORS[k], alpha=0.4, label=f"label {k}")
        for k in [0, 1, 2, 3, "silent"]
    ]
    ax.legend(
        handles=tracker_handles + label_patches,
        labels=tracker_labels + [p.get_label() for p in label_patches],
        loc="upper right",
        fontsize=8,
    )
    fig.tight_layout()
    plt.show()

Visualization skipped.


In [16]:
if decision == "ABANDON":
    print("Per-trajectory preview skipped.")
else:
    preview_selections = [
        sel for sel in traj_selections if sel["traj"].id != SILENT_TRAJECTORY_ID
    ][:PREVIEW_COUNT]
    if not preview_selections:
        raise RuntimeError("No non-silent trajectory found.")

    print(
        f"Previewing {len(preview_selections)} non-silent trajectories "
        f"(indices {preview_selections[0]['index']}–{preview_selections[-1]['index']})"
    )

    for sel in preview_selections:
        traj = sel["traj"]
        start = float(sel["start"])
        duration = float(traj.dur_tot)
        label = trajectory_label(traj)

        y_seg, _ = librosa.load(audio_path, sr=sr, mono=True, offset=start, duration=duration)
        seg_times_pyin, seg_pyin = slice_f0(pyin_times, pyin_f0, start, duration)
        idtap_times, idtap_freqs = idtap_contour(traj)
        traj_mae = summary.loc[summary["traj_index"] == sel["index"], "pyin_mae_cents"].iloc[0]

        print(
            f"\n#{sel['index']} {traj.name_} (label={label}, {duration:.2f}s, "
            f"MAE={traj_mae:.1f} cents)"
        )
        display(Audio(data=y_seg, rate=sr))

        fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
        ax_hz, ax_cents = axes

        ax_hz.plot(idtap_times, idtap_freqs, color="cyan", linewidth=2.5, label="IDTAP")
        ax_hz.plot(seg_times_pyin, seg_pyin, color="#E45756", linewidth=1.5, linestyle="--", label="pyin")
        ax_hz.set_ylabel("Frequency (Hz)")
        ax_hz.set_title(f"#{sel['index']} {traj.name_} (label={label}, MAE={traj_mae:.1f}c)")
        ax_hz.legend(loc="upper right")
        ax_hz.set_xlim(0, duration)

        ax_cents.plot(
            idtap_times,
            freq_to_cents(idtap_freqs, tonic_hz),
            color="cyan",
            linewidth=2.5,
            label="IDTAP",
        )
        ax_cents.plot(
            seg_times_pyin,
            freq_to_cents(seg_pyin, tonic_hz),
            color="#E45756",
            linewidth=1.5,
            linestyle="--",
            label="pyin",
        )
        ax_cents.set_ylabel("Cents relative to tonic")
        ax_cents.set_xlabel("Time within trajectory (s)")
        ax_cents.legend(loc="upper right")
        ax_cents.set_xlim(0, duration)

        fig.tight_layout()
        plt.show()

Per-trajectory preview skipped.


In [17]:
print("=== Worst-matching trajectories (best config) ===")
display(
    best_per_traj[best_per_traj["label"] != "silent"]
    .sort_values("mae_cents", ascending=False)
    .head(20)
)

print("\n=== Thresholds ===")
print(f"  ABANDON if mean MAE > {MAE_ABANDON_CENTS:.0f} cents or voiced < {VOICED_MIN_PCT:.0f}%")
print(f"  CAUTION if mean MAE > {MAE_CAUTION_CENTS:.0f} cents or good-traj fraction < {100*GOOD_TRAJ_FRACTION:.0f}%")
print(f"  Final decision: {decision}")

=== Worst-matching trajectories (best config) ===


,traj_index,idtap_name,label,duration,voiced_pct,mae_cents
428,428,Fixed,0,0.139130,100.000000,1924.261229
36,36,Bend: Simple Multiple,1,0.367849,93.548387,1911.878465
429,429,Bend: Sloped Start,2,0.200000,35.294118,1910.147990
256,256,Fixed,0,0.791304,44.927536,1903.266605
438,438,Bend: Simple Multiple,1,2.547826,76.363636,1825.396862
512,512,Bend: Sloped Start,2,0.147826,15.384615,1764.232708
511,511,Fixed,0,0.156522,64.285714,1698.427896
475,475,Fixed,0,0.739130,84.126984,1650.126009
151,151,Bend: Simple Multiple,1,0.765217,89.393939,1603.430987
367,367,Bend: Simple Multiple,1,0.608696,75.000000,1558.578973



=== Thresholds ===
  ABANDON if mean MAE > 60 cents or voiced < 55%
  CAUTION if mean MAE > 40 cents or good-traj fraction < 60%
  Final decision: ABANDON
